In [38]:
!pip install -q torch

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for ch, i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: (''.join([itos[i] for i in l]))

print(encode('Hello, World!'))
print(decode([20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]))

[20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Hello, World!


In [6]:
print(len(text))
batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

1115394


In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [8]:
n = int(len(text)*0.9)
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
print(train_data[:block_size+1])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t:t+1]
  print(context, 'the next int', target)

tensor([18]) the next int tensor([47])
tensor([18, 47]) the next int tensor([56])
tensor([18, 47, 56]) the next int tensor([57])
tensor([18, 47, 56, 57]) the next int tensor([58])
tensor([18, 47, 56, 57, 58]) the next int tensor([1])
tensor([18, 47, 56, 57, 58,  1]) the next int tensor([15])
tensor([18, 47, 56, 57, 58,  1, 15]) the next int tensor([47])
tensor([18, 47, 56, 57, 58,  1, 15, 47]) the next int tensor([58])


In [11]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(0, len(data) - block_size -1, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

xb, yb = get_batch('train')
print(xb.shape)
xb, yb

torch.Size([4, 8])


(tensor([[53, 59,  6,  1, 58, 56, 47, 40],
         [49, 43, 43, 54,  1, 47, 58,  1],
         [13, 52, 45, 43, 50, 53,  8,  0],
         [ 1, 39,  1, 46, 53, 59, 57, 43]]),
 tensor([[59,  6,  1, 58, 56, 47, 40, 59],
         [43, 43, 54,  1, 47, 58,  1, 58],
         [52, 45, 43, 50, 53,  8,  0, 26],
         [39,  1, 46, 53, 59, 57, 43,  0]]))

In [17]:
n_embd = 32

In [19]:
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape # (4, 8)
    tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)
    x = tok_emb + pos_emb # (B, T, n_embd) + (T, n_embd) = (B, T, n_embd)
    logits = self.lm_head(x) 


    if targets is None:
      loss = None
    
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(idx)
      logits = logits[:, -1, :] # taking the last time step
      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1) # (Batch_size, 1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [20]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [21]:
for iter in range(max_iters):
  
  if iter % eval_interval == 0:
    losses = estimate_loss()
    print(f'Step {iter} losses: Train loss {losses['train']:.4f}, Val loss {losses['val']:.4f}')
  
  xb, yb = get_batch('train')
  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

Step 0 losses: Train loss 2.7181, Val loss 2.7339
Step 300 losses: Train loss 2.6657, Val loss 2.6837
Step 600 losses: Train loss 2.6656, Val loss 2.6665
Step 900 losses: Train loss 2.6398, Val loss 2.6767
Step 1200 losses: Train loss 2.6013, Val loss 2.6225
Step 1500 losses: Train loss 2.6508, Val loss 2.6213
Step 1800 losses: Train loss 2.6037, Val loss 2.5979
Step 2100 losses: Train loss 2.5881, Val loss 2.5674
Step 2400 losses: Train loss 2.5486, Val loss 2.5872
Step 2700 losses: Train loss 2.5706, Val loss 2.5641


In [15]:
print(decode( m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist() ))


PZq-Bo br.
F'xir?Qppever FM
Go imathuiged ad LCubll.cotif HYanCHus o ay&A'Ku,-t;veapayJOLth, the;isticr?KBPd g sf w!
CHer pheeay INCxalo; d inonQbrDYd, s ine th engs ctalyeaba beligryin th wa elacj& by fy FrJt,SfRKI'DP


NC
P t, geac;mor nq,Stowen:
YSl, P gYSay, stirgOCAnes emuBYor.

IO:THevoHEbe.
ba!
NGSV
Pd t theasugn;it'mazKu.galy,:qfMGS?wiAPinPy
Esh:3SB.i'rwn ceontheast mulonbeg!

TXxctlsst?wh fxHTE3Qn:MMCJFDZ teapayse.
Jk:
RTothoonorceph ors,

Ivithe maneem, n.R fadane Y.
DUu:KHAiR:

NCOLIn


In [16]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [17]:
xbow = torch.zeros((B, T, C))
for b in range(B):
  for t in range(T):
    xprev = x[b, :t+1]
    xbow[b, t] = torch.mean(xprev, 0)
x[0], xbow[0]

(tensor([[-0.2265, -0.5562],
         [-1.0454, -0.9823],
         [ 1.8284, -0.7733],
         [-0.1286,  0.6495],
         [-0.2441,  0.7455],
         [ 0.2410, -0.9125],
         [-0.4262, -0.7017],
         [-0.6610,  1.4382]]),
 tensor([[-2.2654e-01, -5.5625e-01],
         [-6.3595e-01, -7.6925e-01],
         [ 1.8550e-01, -7.7060e-01],
         [ 1.0698e-01, -4.1559e-01],
         [ 3.6753e-02, -1.8337e-01],
         [ 7.0791e-02, -3.0489e-01],
         [-2.0322e-04, -3.6158e-01],
         [-8.2802e-02, -1.3660e-01]]))

In [18]:
wei = torch.tril(torch.ones(T, T))
wei = wei / torch.sum(wei, 1, keepdim=True)
xbow2 = wei @ x 
xbow[0], xbow2[0]

(tensor([[-2.2654e-01, -5.5625e-01],
         [-6.3595e-01, -7.6925e-01],
         [ 1.8550e-01, -7.7060e-01],
         [ 1.0698e-01, -4.1559e-01],
         [ 3.6753e-02, -1.8337e-01],
         [ 7.0791e-02, -3.0489e-01],
         [-2.0322e-04, -3.6158e-01],
         [-8.2802e-02, -1.3660e-01]]),
 tensor([[-2.2654e-01, -5.5625e-01],
         [-6.3595e-01, -7.6925e-01],
         [ 1.8550e-01, -7.7060e-01],
         [ 1.0698e-01, -4.1559e-01],
         [ 3.6753e-02, -1.8337e-01],
         [ 7.0791e-02, -3.0489e-01],
         [-2.0323e-04, -3.6158e-01],
         [-8.2802e-02, -1.3660e-01]]))

In [20]:
tril = torch.tril(torch.ones((T, T)))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True